In [1]:
import torch

# ตรวจสอบและใช้งาน GPU ของชิป M4
device = torch.device("mps") if torch.backends.mps.is_available() else torch.device("cpu")
print(f"รันโมเดลบน: {device}")

รันโมเดลบน: mps


In [2]:
import numpy as np
from scipy.io import loadmat

ai_model = 'cnn'
snr = 0
scenario = 'O1'
frequency = 140
antennas = 64

path = f'../DeepMIMO/DeepMIMO/DeepMIMO_dataset/SNR{snr}dB_{scenario}_{frequency}_Ant{antennas}/'
d1 = loadmat(path+'channel1.mat')['a']
d2 = loadmat(path+'channel2.mat')['b']
d3 = loadmat(path+'channel3.mat')['c']

data = np.concatenate((d1, d2, d3), axis=2).transpose(2, 0, 1)

d_r = data.real.reshape(-1, 1, 64, 32)
d_i = data.imag.reshape(-1, 1, 64, 32)

X = np.concatenate((d_r, d_i), axis=1)
mean = np.mean(X, axis=0)
std = np.std(X, axis=0)
X = (X - mean) / (std + 1e-8) # 1e-8 prevents division by zero

print("Data normalization complete. Features now have Mean ≈ 0 and Std ≈ 1.")

# Load Label (One-hot encoding)
y = loadmat(path+'DLCB_output.mat')['onehot_label']
se_data = loadmat(path+'rate_ave.mat')['DL_output']

print(f"Input shape: {X.shape}")
print(f"Label shape: {y.shape}")

from torch.utils.data import DataLoader, TensorDataset

split_idx = int(len(X) * 0.7)

# Training set: The first 70% of the user path
X_train, X_test = X[:split_idx], X[split_idx:]
y_train, y_test = y[:split_idx], y[split_idx:]

se_test = se_data[split_idx:]

print(f"Data Split Complete:")
print(f" - Training samples: {len(X_train)} (First 70%)")
print(f" - Testing samples:  {len(X_test)} (Last 30% - Future Path)")
print(f" - SE test shape: {se_test.shape}")

train_ds = TensorDataset(torch.tensor(X_train).float(), torch.tensor(y_train).float())
test_ds = TensorDataset(torch.tensor(X_test).float(), torch.tensor(y_test).float())

train_loader = DataLoader(train_ds, batch_size=128, shuffle=True)
test_loader = DataLoader(test_ds, batch_size=128, shuffle=False)

Data normalization complete. Features now have Mean ≈ 0 and Std ≈ 1.
Input shape: (53467, 2, 64, 32)
Label shape: (53467, 64)
Data Split Complete:
 - Training samples: 37426 (First 70%)
 - Testing samples:  16041 (Last 30% - Future Path)
 - SE test shape: (16041, 64)


In [ ]:
import torch
import torch.nn as nn

class BeamPredictionViT(nn.Module):
    def __init__(self, in_channels=2, img_size=(64, 32), patch_size=8, 
                 d_model=64, num_heads=4, num_layers=2, n_beams=64, dropout_rate=0.2):
        super(BeamPredictionViT, self).__init__()
        
        # 1. คำนวณจำนวนชิ้นส่วน (Patches)
        # ภาพขนาด 64x32 หั่นเป็นชิ้นละ 8x8 จะได้ = (64/8) * (32/8) = 8 * 4 = 32 Patches
        self.num_patches = (img_size[0] // patch_size) * (img_size[1] // patch_size)
        
        # [ Box 1 & 2: Patch Extraction + Linear Embedding ]
        # ทริคของ PyTorch: ใช้ Conv2d ในการหั่น Patch และบีบอัดมิติ (d_model) ในขั้นตอนเดียว!
        self.patch_embed = nn.Conv2d(in_channels, d_model, kernel_size=patch_size, stride=patch_size)
        
        # [ Box 3: CLS Token ]
        # สร้าง "ตัวแทนประจำห้อง" เพื่อให้มันเรียนรู้ภาพรวมทั้งหมดของ 32 Patches 
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        
        # [ Box 4: Learnable Positional Encoding ]
        # ViT นิยมใช้ Positional Encoding แบบให้โมเดลฝึกฝนเอง (Learnable) แทนสมการ Sin/Cos
        # +1 เพราะต้องเผื่อที่นั่งให้ CLS Token ด้วย
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches + 1, d_model))
        self.pos_drop = nn.Dropout(p=dropout_rate)
        
        # [ Box 5: Transformer Encoder Block ]
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, 
            nhead=num_heads, 
            dim_feedforward=d_model*4, 
            dropout=dropout_rate,
            batch_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)
        
        # [ Box 6: Output Classifier ]
        self.fc = nn.Linear(d_model, n_beams)

    def forward(self, x):
        # [ Input ] Shape: (Batch, 2, 64, 32)
        B = x.shape[0]
        
        # หั่นภาพเป็น Patch และบีบอัดมิติ
        x = self.patch_embed(x)        # Shape: (Batch, d_model, 8, 4)
        x = x.flatten(2)               # Shape: (Batch, d_model, 32)
        x = x.transpose(1, 2)          # Shape: (Batch, 32, d_model) (เรียงลำดับใหม่ให้ Transformer อ่านง่าย)
        
        # นำ CLS Token มาต่อคิวไว้ข้างหน้าสุด (Index 0)
        cls_tokens = self.cls_token.expand(B, -1, -1)  # ขยาย CLS ให้เท่ากับจำนวน Batch
        x = torch.cat((cls_tokens, x), dim=1)          # Shape กลายเป็น (Batch, 33, d_model)
        
        # บวก Positional Encoding
        x = x + self.pos_embed
        x = self.pos_drop(x)
        
        # ส่งเข้า Multi-Head Attention
        x = self.transformer_encoder(x) # Shape: (Batch, 33, d_model)
        
        # [ Output Extraction ] 
        # ไม่ดึง Index -1 แล้ว แต่เราจะดึง Index 0 (ตำแหน่งของ CLS Token) ออกมาฟันธง
        cls_out = x[:, 0, :] # Shape: (Batch, d_model)
        
        # ทำนายผล Beam
        out = self.fc(cls_out) # Shape: (Batch, 64)
        
        return out

In [ ]:
from sklearn.model_selection import train_test_split

model = BeamPredictionViT(
    in_channels=2, 
    img_size=(64, 32), 
    patch_size=8, 
    d_model=64, 
    num_heads=4, 
    num_layers=2, 
    n_beams=64, 
    dropout_rate=0.2
).to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=5, factor=0.5)
criterion = nn.CrossEntropyLoss()

params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total Trainable Params: {params}')

In [ ]:
import torch

model.eval()

dummy_input = torch.randn(1, 2, 64, 32).to(device)

onnx_file_path = "low_level_" + ai_model.lower()+".onnx"

torch.onnx.export(
    model,
    dummy_input,
    onnx_file_path,
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={
        'input': {0: 'batch_size'},
        'output': {0: 'batch_size'}
    }
)

print(f"Model successfully exported to {onnx_file_path}!")

In [ ]:
from torchview import draw_graph
import torch


dummy_input = torch.randn(1, 2, 64, 32).to(device)

model_graph = draw_graph(model, input_size=(1, 2, 64, 32), depth=1, expand_nested=False)
model_graph.visual_graph.render(f"high_level_{ai_model.lower()}", format="png")

In [ ]:
epochs = 100
best_loss = float('inf')
train_losses = []
val_losses = []
save_path = './best_models/'

for epoch in range(epochs):
    epoch_loss = 0.0
    model.train()
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        # Use torch.max to find the best index of beam for CrossEntropyLoss
        loss = criterion(outputs, torch.max(labels, 1)[1])
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
        
    avg_train_loss = epoch_loss / len(train_loader)
    train_losses.append(avg_train_loss)
    
    val_loss = 0.0
    model.eval()
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, torch.max(labels, 1)[1])
            val_loss += loss.item()
    
    avg_val_loss = val_loss / len(test_loader)
    val_losses.append(avg_val_loss)
        
    scheduler.step(avg_val_loss)
        
    if (epoch + 1) % 10 == 0:
        print(f'Epoch [{epoch+1}/{epochs}], Avg Loss: {avg_train_loss:.4f}, Val Loss: {avg_val_loss:.4f}, LR: {optimizer.param_groups[0]["lr"]:.6f}')

    if avg_val_loss < best_loss:
        best_loss = avg_val_loss
        torch.save(model.state_dict(), save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth')
        print(f"--> Saved better model at Epoch {epoch+1} with Loss: {best_loss:.4f}")

        

In [ ]:
from datetime import datetime as dt
import evaluate as ev
import visualizer as vis
import numpy as np

# 1. โหลดโมเดลที่ดีที่สุด
model.load_state_dict(torch.load(
    save_path+f'best_{ai_model.lower()}_model_snr{snr}_{scenario}_{frequency}ghz_{antennas}ant.pth'))
model.eval() # อย่าลืมเซ็ตโมเดลให้อยู่ในโหมด Evaluate

eval_datetime = dt.now().strftime("%Y-%m-%d %H:%M:%S")

ds_config = {
    'snr': snr,
    'scenario': scenario,
    'frequency': frequency,
    'antennas': antennas
}

# 2. ประเมินผลผ่านไฟล์ evaluate.py
mimo_results = ev.evaluate_performance(
    model,
    test_loader,
    device,
    criterion,
    ai_model,
    ds_config,
    eval_datetime
)

# 3. วาดกราฟพื้นฐาน
# หากจะพล็อตกราฟ Loss อย่าลืมส่ง val_losses และ eval_datetime เข้าไปด้วย
# vis.plot_training_loss(train_losses, val_losses, mimo_results, ds_config, eval_datetime)

vis.plot_confusion_matrix(
    mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)

vis.plot_beam_tracking(
    mimo_results['all_actuals'], mimo_results['all_preds'], ai_model, ds_config, eval_datetime)

# 4. เตรียมข้อมูล Spectral Efficiency (SE) จากผลลัพธ์การทำนายจริง (ลบโค้ดสุ่มข้อมูลทิ้ง)
preds_array = np.array(mimo_results['all_preds'])
actuals_array = np.array(mimo_results['all_actuals'])

num_users = len(preds_array)
user_indices = np.arange(num_users)

predicted_se = se_test[user_indices, preds_array]
optimal_se = se_test[user_indices, actuals_array]

plot_limit = 200

# 5. วาดกราฟ SE และบันทึกผลลง CSV
vis.plot_se_tracking(
    user_indices=user_indices[:plot_limit],
    optimal_se=optimal_se[:plot_limit],
    predicted_se=predicted_se[:plot_limit],
    model_name=ai_model,
    ds_config=ds_config,
    eval_datetime=eval_datetime
)

vis.save_se_summary_to_csv(
    mimo_results=mimo_results,
    se_test=se_test,
    model_name=ai_model,
    ds_config=ds_config,
    eval_datetime=eval_datetime
)